In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# ==============================================================================
# 1. DATEN LADEN & ZIELVARIABLE DISKRETISIEREN
# ==============================================================================
df_base = pd.read_pickle("df_model_base_no_geo.pkl")


def assign_quality_class(rating):
    if rating >= 4.90:
        return 2
    elif rating >= 4.60:
        return 1
    else:
        return 0


y1 = df_base["review_scores_rating"].apply(assign_quality_class)

# Leakage- und redundante Spalten sicher entfernen
drop_cols1 = ["review_scores_rating"]
for c in [
    "review_scores_location",
    "review_neighbourhood_rating",
    "neighbourhood_cleansed",
]:
    if c in df_base.columns:
        drop_cols1.append(c)

X1 = df_base.drop(columns=drop_cols1)

# ==============================================================================
# 2. STRATIFIZIERTER SPLIT & BALANCIERUNG
# ==============================================================================
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.20, random_state=42, stratify=y1
)

sample_weights1 = compute_sample_weight(class_weight="balanced", y=y1_train)

# ==============================================================================
# 3. 5-FOLD STRATIFIED GRIDSEARCHCV
# ==============================================================================
param_grid = {
    "max_depth": [2, 3],
    "learning_rate": [0.03, 0.05],
    "reg_alpha": [0.1, 1.0, 5.0],
    "reg_lambda": [1.0, 5.0, 10.0],
    "n_estimators": [150, 200],
}

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_m1 = GridSearchCV(
    estimator=XGBClassifier(
        objective="multi:softprob", eval_metric="mlogloss", random_state=42
    ),
    param_grid=param_grid,
    cv=cv_strat,
    scoring="roc_auc_ovr",
    n_jobs=-1,
    verbose=0,
)

grid_m1.fit(X1_train, y1_train, sample_weight=sample_weights1)
best_m1 = grid_m1.best_estimator_

# ==============================================================================
# 4. EVALUATION AUF DEM 20 % HOLDOUT-TESTSET
# ==============================================================================
preds_m1 = best_m1.predict(X1_test)
probs_m1 = best_m1.predict_proba(X1_test)

print("=" * 80)
print(f"=== MODELL 1 (OHNE GEO) - BESTE PARAMETER: {grid_m1.best_params_} ===")
print("-" * 80)
print(f"Test Accuracy: {accuracy_score(y1_test, preds_m1):.4f}")
print(f"Test ROC-AUC:  {roc_auc_score(y1_test, probs_m1, multi_class='ovr'):.4f}")
print("-" * 80)
print(
    classification_report(
        y1_test,
        preds_m1,
        target_names=["Klasse 0 (Riskant)", "Klasse 1 (Solide)", "Klasse 2 (Premium)"],
    )
)

=== MODELL 1 (OHNE GEO) - BESTE PARAMETER: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'reg_alpha': 5.0, 'reg_lambda': 1.0} ===
--------------------------------------------------------------------------------
Test Accuracy: 0.5643
Test ROC-AUC:  0.7783
--------------------------------------------------------------------------------
                    precision    recall  f1-score   support

Klasse 0 (Riskant)       0.45      0.77      0.57       231
 Klasse 1 (Solide)       0.65      0.38      0.48       592
Klasse 2 (Premium)       0.59      0.70      0.64       437

          accuracy                           0.56      1260
         macro avg       0.56      0.62      0.56      1260
      weighted avg       0.59      0.56      0.55      1260



In [5]:
import numpy as np
import pandas as pd
import shap

# ==============================================================================
# 1. TABELLE 1: FEATURE IMPORTANCE (GAIN) AUS DEM BESTEHENDEN BEST_M1
# ==============================================================================
print("\n" + "=" * 90)
print("--- TABELLE 1: FEATURE IMPORTANCE (MODELL 1 OHNE GEO, SORTIERT) ---")
print("-" * 90)

imp_df = pd.DataFrame(
    {"Feature": X1_train.columns, "Importance": best_m1.feature_importances_}
).sort_values(by="Importance", ascending=False)

imp_df["Importance_Prozent"] = (imp_df["Importance"] / imp_df["Importance"].sum()) * 100

with pd.option_context("display.max_rows", None, "display.float_format", "{:.6f}".format):
    print(
        imp_df.to_string(
            index=False,
            formatters={
                "Importance": "{:.5f}".format,
                "Importance_Prozent": "{:.2f}%".format,
            },
        )
    )

# ==============================================================================
# 2. TABELLE 2: SHAP-WERTE & WIRKUNGSRICHTUNG (PREMIUM KLASSE 2)
# ==============================================================================
print("\n" + "=" * 90)
print("--- TABELLE 2: SHAP-WERTE & WIRKUNGSRICHTUNG (PREMIUM KLASSE 2) ---")
print("-" * 90)

# TreeExplainer direkt auf best_m1 anwenden
explainer = shap.TreeExplainer(best_m1)
raw_shap_values = explainer.shap_values(X1_test)

# Kompatibilität für verschiedene SHAP-Output-Formate (3D-Array vs. Liste von 2D-Arrays)
if isinstance(raw_shap_values, np.ndarray) and len(raw_shap_values.shape) == 3:
    shap_values = [raw_shap_values[:, :, c] for c in range(raw_shap_values.shape[2])]
else:
    shap_values = raw_shap_values

shap_summary_data = []

for i, col in enumerate(X1_train.columns):
    mean_abs_shap_c0 = np.mean(np.abs(shap_values[0][:, i]))
    mean_abs_shap_c1 = np.mean(np.abs(shap_values[1][:, i]))
    mean_abs_shap_c2 = np.mean(np.abs(shap_values[2][:, i]))

    feat_vector = pd.Series(X1_test[col].values)
    shap_vector = pd.Series(shap_values[2][:, i])
    corr_premium = feat_vector.corr(shap_vector)

    if pd.isna(corr_premium) or mean_abs_shap_c2 < 1e-5:
        direction = "Neutral / Kein Effekt"
    elif corr_premium > 0.05:
        direction = "POSITIV (Höhere Werte fördern Premium)"
    elif corr_premium < -0.05:
        direction = "NEGATIV (Höhere Werte senken Premium)"
    else:
        direction = "Geringer / Nicht-linearer Effekt"

    shap_summary_data.append(
        {
            "Feature": col,
            "SHAP_Riskant_C0": mean_abs_shap_c0,
            "SHAP_Solide_C1": mean_abs_shap_c1,
            "SHAP_Premium_C2": mean_abs_shap_c2,
            "Richtung (Klasse 2)": direction,
        }
    )

shap_df = pd.DataFrame(shap_summary_data).sort_values(
    by="SHAP_Premium_C2", ascending=False
)

with pd.option_context(
    "display.max_rows", None, "display.width", 1200, "display.float_format", "{:.6f}".format
):
    print(shap_df.to_string(index=False))

print("=" * 90)


--- TABELLE 1: FEATURE IMPORTANCE (MODELL 1 OHNE GEO, SORTIERT) ---
------------------------------------------------------------------------------------------
                         Feature Importance Importance_Prozent
                    is_superhost    0.24591             24.59%
  calculated_host_listings_count    0.21890             21.89%
             is_instant_bookable    0.10954             10.95%
                   amenity_count    0.03435              3.43%
                 price_per_guest    0.02996              3.00%
                availability_365    0.02790              2.79%
                beds_per_bedroom    0.02712              2.71%
                  minimum_nights    0.02500              2.50%
                           price    0.02397              2.40%
          room_type_Private room    0.02352              2.35%
            host_acceptance_rate    0.02263              2.26%
           availability_ratio_30    0.02189              2.19%
              host_re

In [2]:
# ==============================================================================
# 1. DATEN LADEN & ZIELVARIABLE DISKRETISIEREN
# ==============================================================================
df_ohe = pd.read_pickle("df_model_neighbourhood_ohe.pkl")

y2 = df_ohe["review_scores_rating"].apply(assign_quality_class)

drop_cols2 = ["review_scores_rating"]
for c in [
    "review_scores_location",
    "review_neighbourhood_rating",
    "neighbourhood_cleansed",
]:
    if c in df_ohe.columns:
        drop_cols2.append(c)

X2 = df_ohe.drop(columns=drop_cols2)

# ==============================================================================
# 2. STRATIFIZIERTER SPLIT & BALANCIERUNG
# ==============================================================================
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42, stratify=y2
)

sample_weights2 = compute_sample_weight(class_weight="balanced", y=y2_train)

# ==============================================================================
# 3. 5-FOLD STRATIFIED GRIDSEARCHCV
# ==============================================================================
grid_m2 = GridSearchCV(
    estimator=XGBClassifier(
        objective="multi:softprob", eval_metric="mlogloss", random_state=42
    ),
    param_grid=param_grid,
    cv=cv_strat,
    scoring="roc_auc_ovr",
    n_jobs=-1,
    verbose=0,
)

grid_m2.fit(X2_train, y2_train, sample_weight=sample_weights2)
best_m2 = grid_m2.best_estimator_

# ==============================================================================
# 4. EVALUATION AUF DEM 20 % HOLDOUT-TESTSET
# ==============================================================================
preds_m2 = best_m2.predict(X2_test)
probs_m2 = best_m2.predict_proba(X2_test)

print("=" * 80)
print(f"=== MODELL 2 (OHE KIEZE) - BESTE PARAMETER: {grid_m2.best_params_} ===")
print("-" * 80)
print(f"Test Accuracy: {accuracy_score(y2_test, preds_m2):.4f}")
print(f"Test ROC-AUC:  {roc_auc_score(y2_test, probs_m2, multi_class='ovr'):.4f}")
print("-" * 80)
print(
    classification_report(
        y2_test,
        preds_m2,
        target_names=["Klasse 0 (Riskant)", "Klasse 1 (Solide)", "Klasse 2 (Premium)"],
    )
)

=== MODELL 2 (OHE KIEZE) - BESTE PARAMETER: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'reg_alpha': 5.0, 'reg_lambda': 1.0} ===
--------------------------------------------------------------------------------
Test Accuracy: 0.5659
Test ROC-AUC:  0.7784
--------------------------------------------------------------------------------
                    precision    recall  f1-score   support

Klasse 0 (Riskant)       0.45      0.77      0.57       231
 Klasse 1 (Solide)       0.65      0.38      0.48       592
Klasse 2 (Premium)       0.59      0.71      0.65       437

          accuracy                           0.57      1260
         macro avg       0.57      0.62      0.57      1260
      weighted avg       0.60      0.57      0.55      1260



In [4]:
# ==============================================================================
# 1. DATEN LADEN & ZIELVARIABLE DISKRETISIEREN
# ==============================================================================
df_target = pd.read_pickle("df_model_target_encoded.pkl")

y3 = df_target["review_scores_rating"].apply(assign_quality_class)

drop_cols3 = ["review_scores_rating"]
for c in [
    "review_scores_location",
    "review_neighbourhood_rating",
    "neighbourhood_cleansed",
]:
    if c in df_target.columns:
        drop_cols3.append(c)

X3 = df_target.drop(columns=drop_cols3)

# ==============================================================================
# 2. STRATIFIZIERTER SPLIT & BALANCIERUNG
# ==============================================================================
X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.20, random_state=42, stratify=y3
)

sample_weights3 = compute_sample_weight(class_weight="balanced", y=y3_train)

# ==============================================================================
# 3. 5-FOLD STRATIFIED GRIDSEARCHCV
# ==============================================================================
grid_m3 = GridSearchCV(
    estimator=XGBClassifier(
        objective="multi:softprob", eval_metric="mlogloss", random_state=42
    ),
    param_grid=param_grid,
    cv=cv_strat,
    scoring="roc_auc_ovr",
    n_jobs=-1,
    verbose=0,
)

grid_m3.fit(X3_train, y3_train, sample_weight=sample_weights3)
best_m3 = grid_m3.best_estimator_

# ==============================================================================
# 4. EVALUATION AUF DEM 20 % HOLDOUT-TESTSET
# ==============================================================================
preds_m3 = best_m3.predict(X3_test)
probs_m3 = best_m3.predict_proba(X3_test)

print("=" * 80)
print(f"=== MODELL 3 (TARGET ENCODING) - BESTE PARAMETER: {grid_m3.best_params_} ===")
print("-" * 80)
print(f"Test Accuracy: {accuracy_score(y3_test, preds_m3):.4f}")
print(f"Test ROC-AUC:  {roc_auc_score(y3_test, probs_m3, multi_class='ovr'):.4f}")
print("-" * 80)
print(
    classification_report(
        y3_test,
        preds_m3,
        target_names=["Klasse 0 (Riskant)", "Klasse 1 (Solide)", "Klasse 2 (Premium)"],
    )
)

# ==============================================================================
# 5. VERGLEICHSTABELLE FÜR DAS ERGEBNISKAPITEL
# ==============================================================================
comparison_table = pd.DataFrame(
    {
        "Modell-Spezifikation": [
            "Modell 1: Basis (ohne Geo)",
            "Modell 2: One-Hot-Encoding (Kieze)",
            "Modell 3: Target Encoding (Kiez-Scores)",
        ],
        "Anzahl Features": [X1.shape[1], X2.shape[1], X3.shape[1]],
        "Test-Accuracy": [
            accuracy_score(y1_test, preds_m1),
            accuracy_score(y2_test, preds_m2),
            accuracy_score(y3_test, preds_m3),
        ],
        "Test-ROC-AUC (OVR)": [
            roc_auc_score(y1_test, probs_m1, multi_class="ovr"),
            roc_auc_score(y2_test, probs_m2, multi_class="ovr"),
            roc_auc_score(y3_test, probs_m3, multi_class="ovr"),
        ],
    }
).set_index("Modell-Spezifikation")

print("\n" + "=" * 80)
print("=== GESAMTVERGLEICH DER DREI KLASSIFIKATIONS-SPEZIFIKATIONEN ===")
print("=" * 80)
print(
    comparison_table.to_string(
        formatters={"Test-Accuracy": "{:.4f}".format, "Test-ROC-AUC (OVR)": "{:.4f}".format}
    )
)

=== MODELL 3 (TARGET ENCODING) - BESTE PARAMETER: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'reg_alpha': 5.0, 'reg_lambda': 1.0} ===
--------------------------------------------------------------------------------
Test Accuracy: 0.5627
Test ROC-AUC:  0.7802
--------------------------------------------------------------------------------
                    precision    recall  f1-score   support

Klasse 0 (Riskant)       0.44      0.76      0.56       231
 Klasse 1 (Solide)       0.66      0.38      0.48       592
Klasse 2 (Premium)       0.59      0.71      0.65       437

          accuracy                           0.56      1260
         macro avg       0.56      0.62      0.56      1260
      weighted avg       0.59      0.56      0.55      1260


=== GESAMTVERGLEICH DER DREI KLASSIFIKATIONS-SPEZIFIKATIONEN ===
                                         Anzahl Features Test-Accuracy Test-ROC-AUC (OVR)
Modell-Spezifikation                                           

Segmentierungsversuch für Klassifikationsmodell

In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# ==============================================================================
# 0. DATEN LADEN & ZIELVARIABLE DEFINIEREN
# ==============================================================================
try:
    df_pipeline = df_model_neighbourhood_ohe.copy()
except NameError:
    df_pipeline = pd.read_pickle("df_model_neighbourhood_ohe.pkl")

# Premium-Listing via Top-25%-Quantil
threshold_top_25 = df_pipeline["review_scores_rating"].quantile(0.75)
df_pipeline["is_premium_listing"] = (
    df_pipeline["review_scores_rating"] >= threshold_top_25
).astype(int)

# Spalten bereinigen (Leakage und Zielvariable entfernen)
base_drop_cols = ["review_scores_rating", "is_premium_listing"]
for c in [
    "review_scores_location",
    "review_neighbourhood_rating",
    "neighbourhood_cleansed",
]:
    if c in df_pipeline.columns:
        base_drop_cols.append(c)

# Modellparameter (reduzierte Modellkapazität gegen Overfitting bei kleineren N)
seg_params = {
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
}

# ==============================================================================
# 1. GLOBALES GESAMTMARKT-MODELL (BASELINE)
# ==============================================================================
y_global = df_pipeline["is_premium_listing"]
X_global = df_pipeline.drop(columns=base_drop_cols)

X_g_temp, X_g_test, y_g_temp, y_g_test = train_test_split(
    X_global, y_global, test_size=0.20, random_state=42, stratify=y_global
)
X_g_train, X_g_val, y_g_train, y_g_val = train_test_split(
    X_g_temp, y_g_temp, test_size=0.25, random_state=42, stratify=y_g_temp
)

xgb_global = XGBClassifier(**seg_params)
xgb_global.fit(X_g_train, y_g_train, eval_set=[(X_g_val, y_g_val)], verbose=False)

g_tr_acc = accuracy_score(y_g_train, xgb_global.predict(X_g_train))
g_te_acc = accuracy_score(y_g_test, xgb_global.predict(X_g_test))
g_te_auc = roc_auc_score(y_g_test, xgb_global.predict_proba(X_g_test)[:, 1])

results = [
    {
        "Modell / Segment": "Gesamtmarkt",
        "Stichprobe (N)": len(df_pipeline),
        "Train Accuracy": g_tr_acc,
        "Test Accuracy": g_te_acc,
        "Δ Test Acc": 0.0,
        "Test ROC-AUC": g_te_auc,
        "Δ Test ROC-AUC": 0.0,
    }
]

# ==============================================================================
# 2. SEGMENTIERUNG NACH ZIMMERTYP (ROOM TYPE)
# ==============================================================================
# Identifikation der relevanten One-Hot-Spalten für room_type
room_dummy_cols = [c for c in df_pipeline.columns if c.startswith("room_type_")]

for col in room_dummy_cols:
    seg_name = col.replace("room_type_", "")
    df_sub = df_pipeline[df_pipeline[col] == 1].copy()

    if len(df_sub) < 150:
        continue

    # Alle Zimmertyp-Dummies aus dem Sub-Feature-Set entfernen
    drop_cols_sub = base_drop_cols + room_dummy_cols
    X_sub = df_sub.drop(columns=[c for c in drop_cols_sub if c in df_sub.columns])
    y_sub = df_sub["is_premium_listing"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_sub, y_sub, test_size=0.20, random_state=42, stratify=y_sub
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
    )

    xgb_sub = XGBClassifier(**seg_params)
    xgb_sub.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    tr_acc = accuracy_score(y_train, xgb_sub.predict(X_train))
    te_acc = accuracy_score(y_test, xgb_sub.predict(X_test))
    te_auc = roc_auc_score(y_test, xgb_sub.predict_proba(X_test)[:, 1])

    results.append(
        {
            "Modell / Segment": f"Zimmertyp: {seg_name}",
            "Stichprobe (N)": len(df_sub),
            "Train Accuracy": tr_acc,
            "Test Accuracy": te_acc,
            "Δ Test Acc": te_acc - g_te_acc,
            "Test ROC-AUC": te_auc,
            "Δ Test ROC-AUC": te_auc - g_te_auc,
        }
    )

# ==============================================================================
# 3. SEGMENTIERUNG NACH REISE-MOTIVATION (KURZZEIT VS. LANGZEIT)
# ==============================================================================
df_pipeline["is_langzeit"] = (df_pipeline["minimum_nights"] >= 30).astype(int)

time_segments = {
    "Kurzzeit-Tourismus (< 30 Nächte)": df_pipeline["is_langzeit"] == 0,
    "Langzeit-Wohnen (≥ 30 Nächte)": df_pipeline["is_langzeit"] == 1,
}

for seg_name, condition in time_segments.items():
    df_sub = df_pipeline[condition].copy()

    if len(df_sub) < 100:
        continue

    # Zeitvariablen entfernen, die die Aufteilung definieren
    time_drop_cols = [
        "minimum_nights",
        "min_nights_to_availability_ratio",
        "is_langzeit",
    ]
    drop_cols_sub = base_drop_cols + time_drop_cols
    X_sub = df_sub.drop(columns=[c for c in drop_cols_sub if c in df_sub.columns])
    y_sub = df_sub["is_premium_listing"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_sub, y_sub, test_size=0.20, random_state=42, stratify=y_sub
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
    )

    xgb_sub = XGBClassifier(**seg_params)
    xgb_sub.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    tr_acc = accuracy_score(y_train, xgb_sub.predict(X_train))
    te_acc = accuracy_score(y_test, xgb_sub.predict(X_test))
    te_auc = roc_auc_score(y_test, xgb_sub.predict_proba(X_test)[:, 1])

    results.append(
        {
            "Modell / Segment": f"Reise-Motivation: {seg_name}",
            "Stichprobe (N)": len(df_sub),
            "Train Accuracy": tr_acc,
            "Test Accuracy": te_acc,
            "Δ Test Acc": te_acc - g_te_acc,
            "Test ROC-AUC": te_auc,
            "Δ Test ROC-AUC": te_auc - g_te_auc,
        }
    )

# ==============================================================================
# 4. TABELLARISCHER ERGEBNISVERGLEICH FÜR DIE MASTERARBEIT
# ==============================================================================
df_results = pd.DataFrame(results).set_index("Modell / Segment")

print("\n" + "=" * 95)
print("=== VERGLEICH DER KLASSIFIKATIONSGÜTE: GESAMTMARKT VS. SEGMENTSPEZIFISCHE MODELLE ===")
print("=" * 95)
print(
    df_results.to_string(
        formatters={
            "Stichprobe (N)": "{:,}".format,
            "Train Accuracy": "{:.4f}".format,
            "Test Accuracy": "{:.4f}".format,
            "Δ Test Acc": "{:+.4f}".format,
            "Test ROC-AUC": "{:.4f}".format,
            "Δ Test ROC-AUC": "{:+.4f}".format,
        }
    )
)
print("=" * 95)


=== VERGLEICH DER KLASSIFIKATIONSGÜTE: GESAMTMARKT VS. SEGMENTSPEZIFISCHE MODELLE ===
                                                   Stichprobe (N) Train Accuracy Test Accuracy Δ Test Acc Test ROC-AUC Δ Test ROC-AUC
Modell / Segment                                                                                                                     
Gesamtmarkt                                                 6,298         0.8129        0.7460    +0.0000       0.7695        +0.0000
Zimmertyp: Entire home/apt                                  4,546         0.8309        0.7615    +0.0155       0.7822        +0.0127
Zimmertyp: Private room                                     1,594         0.8326        0.6959    -0.0501       0.7593        -0.0101
Reise-Motivation: Kurzzeit-Tourismus (< 30 Nächte)          5,675         0.8059        0.7683    +0.0223       0.7911        +0.0217
Reise-Motivation: Langzeit-Wohnen (≥ 30 Nächte)               623         0.9437        0.6560    -0.0900    